# The Dengue Exercise - Lane B (worked solution / captured agent output)

**SISMID 2026 - Day 1, 11:00.** Map a Google search signal onto reported dengue cases
in Mexico, fit on the early years, and honestly validate on the later years.

> Every cell here is an example of what a coding agent (Codex, Claude Code, or
> Antigravity CLI) produces from the prompts in the **Lane A** notebook / on the slides.
> Run it top to bottom as a backup, or read it to see what good output looks like.

Data: `MX_Dengue_trends.csv` - monthly, Mexico, 2004-2011. `Dengue CDC` = reported
cases (ground truth); `dengue`, `sintomas de dengue`, `mosquito`, `dengue sintomas` =
Google search interest. **Train** on 2004-2006 (first 36 months), **validate** on 2007-2011.


## Step 1: visualize (look before you model)

Plot cases over time, overlay the `dengue` search series on a second axis, and print the
correlation. Do they move together? Any lead/lag, gaps, or spikes?


In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

path = next(p for p in ['../data/MX_Dengue_trends.csv','data/MX_Dengue_trends.csv',
                        'MX_Dengue_trends.csv'] if os.path.exists(p))
df = pd.read_csv(path); df['Date'] = pd.to_datetime(df['Date'])
df.head()


In [ ]:
fig, ax1 = plt.subplots(figsize=(10,4))
ax1.plot(df['Date'], df['Dengue CDC'], 'k-', label='dengue cases (CDC)')
ax1.set_xlabel('date'); ax1.set_ylabel('dengue cases')
ax2 = ax1.twinx()
ax2.plot(df['Date'], df['dengue'], color='tab:blue', alpha=0.6, label='"dengue" search')
ax2.set_ylabel('search interest')
plt.title('Dengue cases vs "dengue" search interest, Mexico'); fig.tight_layout(); plt.show()
print('correlation(cases, dengue search):', round(df['Dengue CDC'].corr(df['dengue']), 3))


## Step 2: least squares, the honest baseline

Fit `Dengue CDC ~ dengue` on the first 36 months only; report intercept, slope, and
in-sample $R^2$, and overlay the fitted line on the scatter.


In [ ]:
df_train = df.iloc[:36]
mod = LinearRegression().fit(df_train[['dengue']], df_train['Dengue CDC'])
m, b = mod.coef_[0], mod.intercept_
r2 = mod.score(df_train[['dengue']], df_train['Dengue CDC'])
print(f'Dengue CDC = {m:.1f} * dengue + {b:.1f}   (in-sample R^2 = {r2:.2f})')
xs = np.linspace(df_train['dengue'].min(), df_train['dengue'].max(), 50)
plt.figure(figsize=(7,4))
plt.plot(df_train['dengue'], df_train['Dengue CDC'], 'o', label='training months')
plt.plot(xs, m*xs + b, 'r-', label='least-squares fit')
plt.xlabel('"dengue" search interest'); plt.ylabel('dengue cases')
plt.legend(); plt.title('Fit on 2004-2006'); plt.tight_layout(); plt.show()


## Step 3: predict, then judge out of sample

Use the training line to predict 2007-2011 from search **alone** (no refit). Plot
predicted vs actual and report out-of-sample correlation and RMSE.


In [ ]:
df_valid = df.iloc[36:]
pred = m*df_valid['dengue'] + b
plt.figure(figsize=(10,4))
plt.plot(df_valid['Date'], df_valid['Dengue CDC'], 'k-', label='actual cases')
plt.plot(df_valid['Date'], pred, 'b-', label='predicted from search')
plt.xlabel('date'); plt.ylabel('dengue cases'); plt.legend()
plt.title('2007-2011 validation: predicted (search only) vs actual'); plt.tight_layout(); plt.show()
oos_corr = np.corrcoef(df_valid['Dengue CDC'], pred)[0,1]
rmse = np.sqrt(np.mean((df_valid['Dengue CDC'].values - pred.values)**2))
print(f'out-of-sample correlation: {oos_corr:.3f}')
print(f'out-of-sample RMSE      : {rmse:.0f} cases')


## Step 4: discuss - a good fit is not a correct model

The in-sample fit looks strong, but the out-of-sample plot shows where a single search
term drifts from the truth. Both curves partly track the **season**, so correlation can
look good while the mechanism is wrong. This is the Google Flu Trends cautionary tale,
and the reason to validate out of sample and stay humble about a correlation.

**How would you improve it?** More terms, dynamic refitting, and dynamic training - which
build up to **ARGO** on Day 2.


## Reflection

- You built a mini **Google Dengue Trends**: search mapped onto cases, fit on the past,
  validated on the future.
- The one-term line is the honest baseline. **ARGO** (Day 2) improves it with many selected
  terms and dynamic training.
- **Further stretch:** on the `sarahhbellum/Zika-tutorial` data, refit on a rolling window
  and use Lasso across many terms so the useless ones drop out.
